In [25]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
df = wine.frame

df.head()
print(df.shape)
print(df["target"].value_counts().sort_index())

(178, 14)
target
0    59
1    71
2    48
Name: count, dtype: int64


In [26]:
df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [27]:
X = df.drop(columns=["target"])
y = df["target"]

print(X.shape)
print(y.shape)


(178, 13)
(178,)


In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state=42, stratify=y
)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(142, 13)
(36, 13)
(142,)
(36,)


In [29]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

baseline_accuracy = accuracy_score(y_test, baseline_predictions)
print(baseline_accuracy)

0.3888888888888889


In [30]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

model_predictions = model.predict(X_test)
model_accuracy = accuracy_score( y_test, model_predictions)

print("Baseline accuracy:", baseline_accuracy)
print("Model accuracy:", model_accuracy)

Baseline accuracy: 0.3888888888888889
Model accuracy: 0.9722222222222222


In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, model_predictions)

pd.DataFrame(
    cm, 
    index=["Actual 0", "Actual 1", "Actual 2"],
    columns=["Predicted 0", "Predicted 1", "Predicted 2"]
)

print(cm)

[[12  0  0]
 [ 0 14  0]
 [ 0  1  9]]


In [36]:
from sklearn.metrics import classification_report
print(classification_report(y_test, model_predictions))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       0.93      1.00      0.97        14
           2       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36



In [39]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    model, X_train, y_train, cv=cv, scoring="accuracy"
)

print(cv_scores)

print("Mean accuracy", cv_scores.mean())
print("Standard deviation:", cv_scores.std())



[0.93103448 1.         1.         1.         0.96428571]
Mean accuracy 0.9790640394088669
Standard deviation: 0.027713464392187602


In [44]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

forest_scores = cross_val_score(
    forest, 
    X_train,
    y_train,
    scoring="accuracy",
    cv=cv
)

print("Logistic regression:", cv_scores.mean())
print("Random forest:", forest_scores.mean())

Logistic regression: 0.9790640394088669
Random forest: 0.9790640394088669


In [45]:
probabilities = model.predict_proba(X_test)
pd.DataFrame(probabilities, columns=model.classes_, index=X_test.index).head()

,0,1,2
10,0.999671,0.000307,0.000023
134,0.007051,0.540423,0.452526
28,0.992000,0.007664,0.000336
121,0.289967,0.707467,0.002565
62,0.033032,0.962313,0.004655


In [46]:
prediction_review = pd.DataFrame({
    "actual": y_test,
    "predicted": model_predictions,
    "confidence": probabilities.max(axis=1)
})

prediction_review.sort_values("confidence").head()

,actual,predicted,confidence
134,2,1,0.540423
130,2,2,0.600787
38,0,0,0.658539
121,1,1,0.707467
143,2,2,0.867840


In [49]:
probability_table = pd.DataFrame(
    probabilities, 
    columns=model.classes_,
    index=X_test.index
)

probability_table.loc[134]

0    0.007051
1    0.540423
2    0.452526
Name: 134, dtype: float64